# City Stress Index
## Fusing Weather, News Sentiment, and Transit Delays to Forecast Urban Tension

---

Washington DC has a heartbeat. This project listens to it, and predicts when it is about to race.

---

### What This Project Does
Most data projects use one data source. This one fuses three live signals:

| Signal | Source | What it measures |
|---|---|---|
| Weather | Open-Meteo API (free, no key) | Heat, rain, wind pressure on mood |
| News Sentiment | NewsAPI + VADER NLP | Negativity in local headlines |
| Transit Stress | WMATA API (live) + calibrated simulation | Delay rates, overcrowding scores |

These are fused into a single City Stress Index (0-100), then an ML model forecasts tomorrow.

### Stack
requests, sqlite3, pandas, numpy, vaderSentiment, scikit-learn, plotly

### API keys -- read this before running
This notebook needs two free API keys and reads both through a small helper (`get_secret`,
defined in Step 0) that checks, in order:
1. An environment variable (works for local Jupyter, GitHub Actions, Docker, etc.)
2. Colab's `userdata` secrets store (works when running in Google Colab)

**No key is ever hardcoded or printed anywhere in this notebook.** Get free keys at:
- [newsapi.org](https://newsapi.org) -> set as `NEWS_API_KEY`
- [developer.wmata.com](https://developer.wmata.com) -> set as `WMATA_KEY`

In Colab: click the key icon in the left sidebar -> "Secrets" -> add `NEWS_API_KEY` and
`WMATA_KEY`. Locally: `export NEWS_API_KEY=...` and `export WMATA_KEY=...` before launching
Jupyter, or use a `.env` file with `python-dotenv` (make sure `.env` is in `.gitignore`).

Never assign a bare `get_secret(...)` call as the last line of a cell -- Jupyter/Colab
auto-display the return value of the last expression in a cell, which would print your key
into the saved notebook output. Always assign it to a variable first.


## Step 0 -- Install & Import Libraries

In [ ]:
import sqlite3, os, json, time, warnings, random
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)
random.seed(42)
np.random.seed(42)


def get_secret(name):
    """
    Look up an API key by name without ever printing or returning it
    implicitly. Checks an environment variable first (local/CI-friendly),
    then falls back to Colab's userdata secrets store if available.
    Returns None if not found anywhere -- callers should handle that.
    """
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value


print('All libraries loaded!')
print('Run date:', datetime.now().strftime('%Y-%m-%d %H:%M'))


## Step 1 -- City Configuration\nChange TARGET_CITY to any of the five cities below.

In [ ]:
CITIES = {
    'Washington DC': {
        'lat': 38.9072, 'lon': -77.0369,
        'timezone': 'America/New_York',
        'news_query': 'Washington DC metro traffic protest government',
        'transit_baseline_delay': 0.22,
        'population_density': 4515,
    },
}

TARGET_CITY = 'Washington DC'
cfg = CITIES[TARGET_CITY]
print('Target city :', TARGET_CITY)
print('Coordinates :', cfg['lat'], cfg['lon'])


## Step 2 -- Fetch 30 Days of Weather (Open-Meteo API)\nNo API key needed. Returns daily temperature, rain, wind, and weather code.

In [ ]:
LAT = 38.9072
LON = -77.0369
TIMEZONE = 'America/New_York'
DAYS_BACK = 30

WMO_STRESS = {
    0: ('Clear sky', 0.00), 1: ('Mainly clear', 0.05), 2: ('Partly cloudy', 0.10),
    3: ('Overcast', 0.20), 45: ('Fog', 0.30), 48: ('Icy fog', 0.40),
    51: ('Light drizzle', 0.20), 61: ('Rain', 0.40), 63: ('Heavy rain', 0.60),
    71: ('Snow', 0.50), 80: ('Rain showers', 0.45), 95: ('Thunderstorm', 0.85),
    99: ('Severe storm', 1.00),
}

def fetch_weather(lat, lon, timezone, days_back=DAYS_BACK):
    end = datetime.now().strftime('%Y-%m-%d')
    start = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
    print('Fetching weather from', start, 'to', end)
    response = requests.get(
        'https://api.open-meteo.com/v1/forecast',
        params={
            'latitude': lat, 'longitude': lon,
            'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode',
            'timezone': timezone, 'start_date': start, 'end_date': end,
        },
        timeout=15,
    )
    response.raise_for_status()
    data = response.json()['daily']
    df = pd.DataFrame(data)
    df.rename(columns={'time': 'date'}, inplace=True)
    df['date'] = pd.to_datetime(df['date']).dt.normalize()
    df['city'] = TARGET_CITY
    print('Fetched', len(df), 'days of real weather data')
    return df

df_weather = fetch_weather(LAT, LON, TIMEZONE)

df_weather['weather_label'] = df_weather['weathercode'].map(lambda c: WMO_STRESS.get(c, ('Unknown', 0.2))[0])
df_weather['weather_stress'] = df_weather['weathercode'].map(lambda c: WMO_STRESS.get(c, ('Unknown', 0.2))[1])
df_weather['heat_stress'] = (
    (df_weather['temperature_2m_max'] - 22).clip(lower=0) / 16 +
    (-5 - df_weather['temperature_2m_min']).clip(lower=0) / 15
).clip(0, 1)
df_weather['wind_stress'] = (df_weather['windspeed_10m_max'] / 80).clip(0, 1)
df_weather['rain_stress'] = (df_weather['precipitation_sum'] / 60).clip(0, 1)

print('Date range      :', df_weather['date'].min().date(), 'to', df_weather['date'].max().date())
print('Avg Temp Max    :', round(df_weather['temperature_2m_max'].mean(), 1), 'C')
print('Avg Rain        :', round(df_weather['precipitation_sum'].mean(), 1), 'mm')
print('Avg Wind        :', round(df_weather['windspeed_10m_max'].mean(), 1), 'km/h')
print('High Stress Days:', int((df_weather['weather_stress'] >= 0.4).sum()))

display(df_weather[['date', 'weather_label', 'weather_stress', 'heat_stress', 'rain_stress', 'wind_stress']]
    .tail(30)
    .style
    .format({'weather_stress': '{:.2f}', 'heat_stress': '{:.2f}', 'rain_stress': '{:.2f}', 'wind_stress': '{:.2f}'})
    .background_gradient(subset=['weather_stress'], cmap='RdYlGn_r')
    .background_gradient(subset=['rain_stress'],    cmap='Blues')
    .background_gradient(subset=['wind_stress'],    cmap='Purples')
    .background_gradient(subset=['heat_stress'],    cmap='Oranges')
)


## Step 3 -- News Sentiment (VADER NLP)I score daily headlines with VADER, a sentiment analyser built for news and social media.API key required: `NEWS_API_KEY` (see the note in the intro cell for how it's retrieved).

In [ ]:
!pip install -q vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

NEWS_API_KEY = get_secret('NEWS_API_KEY')
if not NEWS_API_KEY:
    raise ValueError(
        'NEWS_API_KEY not found. Set it as an environment variable, or as a Colab '
        'secret named NEWS_API_KEY, then re-run this cell. Get a free key at '
        'https://newsapi.org.'
    )

TARGET_CITY = 'Washington DC'
NEWS_QUERY = '\"Washington DC\" AND (metro OR traffic OR protest OR storm OR shutdown)'
DAYS = 10

analyzer = SentimentIntensityAnalyzer()

def fetch_real_headlines(query, api_key, days=30):
    """Fetch real headlines from NewsAPI one day at a time."""
    by_date = {}
    success_count = 0
    empty_count = 0
    print('Fetching', days, 'days of real headlines for:', TARGET_CITY)
    for d in range(days):
        day = (datetime.now() - timedelta(days=d)).strftime('%Y-%m-%d')
        response = requests.get(
            'https://newsapi.org/v2/everything', timeout=10,
            params={'q': query, 'from': day, 'to': day, 'sortBy': 'popularity',
                    'pageSize': 10, 'apiKey': api_key, 'language': 'en'},
        )
        if response.status_code != 200:
            print(' ', day, 'API error', response.status_code)
            by_date[day] = []
            empty_count += 1
            time.sleep(0.3)
            continue
        articles = response.json().get('articles', [])
        headlines = [a['title'] for a in articles if a.get('title') and a['title'] != '[Removed]']
        by_date[day] = headlines
        if headlines:
            success_count += 1
        else:
            empty_count += 1
        time.sleep(0.2)
    print('Done --', success_count, 'days with data,', empty_count, 'empty days')
    return by_date


In [ ]:
def score_real_headlines(by_date):
    """Run VADER sentiment analysis on every headline."""
    rows = []
    for date_str, headlines in by_date.items():
        if not headlines:
            continue
        scores = [analyzer.polarity_scores(h) for h in headlines]
        neg_avg = np.mean([s['neg'] for s in scores])
        pos_avg = np.mean([s['pos'] for s in scores])
        neu_avg = np.mean([s['neu'] for s in scores])
        comp_avg = np.mean([s['compound'] for s in scores])
        sentiment_stress = neg_avg * 0.6 + (1 - (comp_avg + 1) / 2) * 0.4
        rows.append({
            'date': pd.to_datetime(date_str), 'headline_count': len(headlines),
            'avg_negativity': round(neg_avg, 4), 'avg_positivity': round(pos_avg, 4),
            'avg_neutral': round(neu_avg, 4), 'avg_compound': round(comp_avg, 4),
            'sentiment_stress': round(sentiment_stress, 4), 'sample_headline': headlines[0],
        })
    return pd.DataFrame(rows).sort_values('date').reset_index(drop=True)


In [ ]:
import re

keyword = 'washington dc'

def contains_keyword(text):
    return bool(re.search(keyword, text.lower())) if text else False

def fetch_articles(date_from, date_to, api_key):
    url = 'https://newsapi.org/v2/everything'
    all_articles = []
    page = 1
    while True:
        params = {
            'q': 'Washington DC', 'from': date_from, 'to': date_to,
            'language': 'en', 'sortBy': 'publishedAt', 'pageSize': 100,
            'page': page, 'apiKey': api_key,
        }
        response = requests.get(url, params=params)
        data = response.json()
        if data.get('status') != 'ok':
            print('API error:', data.get('message', 'Unknown error'))
            break
        batch = data.get('articles', [])
        if not batch:
            break
        all_articles.extend(batch)
        total_results = data.get('totalResults', 0)
        print('  Page', page, ': fetched', len(batch), 'articles (total available:', total_results, ')')
        if len(all_articles) >= min(total_results, 1000):
            break
        page += 1
    return all_articles

def analyze(articles):
    results = []
    for article in articles:
        title = article.get('title', '') or ''
        description = article.get('description', '') or ''
        if not (contains_keyword(title) or contains_keyword(description)):
            continue
        text_to_score = (title + '. ' + description).strip()
        scores = analyzer.polarity_scores(text_to_score)
        results.append({
            'date': article.get('publishedAt', '')[:10],
            'source': article.get('source', {}).get('name', ''),
            'title': title, 'compound': scores['compound'], 'pos': scores['pos'],
            'neu': scores['neu'], 'neg': scores['neg'],
            'matched_in': 'title' if contains_keyword(title) else 'description',
            'url': article.get('url', ''),
        })
    return results

date_to = datetime.today()
date_from = date_to - timedelta(days=10)
date_from_str = date_from.strftime('%Y-%m-%d')
date_to_str = date_to.strftime('%Y-%m-%d')
print('Fetching articles from', date_from_str, 'to', date_to_str)

articles = fetch_articles(date_from_str, date_to_str, NEWS_API_KEY)
print('Total raw articles fetched:', len(articles))

results = analyze(articles)
df_news = pd.DataFrame(results)

# Keep a consistent schema even when nothing comes back (bad key, rate limit,
# or an overly strict keyword filter), so downstream SQL joins don't break.
EXPECTED_NEWS_COLUMNS = ['date', 'source', 'title', 'compound', 'pos', 'neu', 'neg', 'matched_in', 'url']
if df_news.empty:
    print('WARNING: 0 articles matched -- check that NEWS_API_KEY is valid and not rate-limited.')
    df_news = pd.DataFrame(columns=EXPECTED_NEWS_COLUMNS)
else:
    df_news = df_news.sort_values('date').reset_index(drop=True)
    print(len(df_news), 'articles passed the keyword filter')
    print('Date range   :', df_news['date'].min(), 'to', df_news['date'].max())
    print('Avg compound :', round(df_news['compound'].mean(), 3))


## Step 4 -- Transit Stress Signal

Transit delays modelled with city-specific baseline delay rates, day-of-week patterns, weather correlation, and live WMATA rail incidents plus bus deviation for today (calibrated simulation for prior days).

API key required: `WMATA_KEY` (see the note in the intro cell for how it's retrieved).

In [ ]:
from IPython.display import display

WMATA_KEY = get_secret('WMATA_KEY')
if not WMATA_KEY:
    raise ValueError(
        'WMATA_KEY not found. Set it as an environment variable, or as a Colab '
        'secret named WMATA_KEY, then re-run this cell. Get a free key at '
        'https://developer.wmata.com.'
    )
_HEADERS = {'api_key': WMATA_KEY}

_BASE = 'https://api.wmata.com'
_RAIL_INCIDENTS = _BASE + '/Incidents.svc/json/Incidents'
_BUS_POSITIONS = _BASE + '/Bus.svc/json/jBusPositions'

DC_TRANSIT_BASELINE = 0.20

def _fetch_rail_incidents():
    """Live Metrorail disruptions. Returns [] on any API error (bad key, network, etc)."""
    try:
        r = requests.get(_RAIL_INCIDENTS, headers=_HEADERS, timeout=8)
        r.raise_for_status()
        return r.json().get('Incidents', [])
    except Exception as e:
        print('  [WMATA] Rail incidents fetch failed:', e)
        return []

def _fetch_bus_deviations(sample_routes=('16Y', 'S2', '42', 'D6', '70')):
    """Mean absolute bus deviation (minutes) across a sample of routes. 0 if unavailable."""
    deviations = []
    for route in sample_routes:
        try:
            r = requests.get(_BUS_POSITIONS, headers=_HEADERS, params={'RouteID': route}, timeout=8)
            r.raise_for_status()
            buses = r.json().get('BusPositions', [])
            deviations.extend(b['Deviation'] for b in buses if 'Deviation' in b)
        except Exception:
            continue
    if not deviations:
        return 0.0
    return sum(abs(d) for d in deviations) / len(deviations)

def _incidents_to_stress(incidents):
    """Convert raw WMATA incident list -> (incident_multiplier, incident_type_label)."""
    if not incidents:
        return 1.0, 'none'
    severity_map = {
        'delay': 1.5, 'single track': 1.8, 'alert': 1.2, 'station closure': 2.0,
        'smoke': 2.2, 'medical': 1.4, 'track work': 1.6, 'power': 1.9,
    }
    multiplier = 1.0
    types_seen = []
    for inc in incidents:
        raw_type = (inc.get('IncidentType') or '').lower()
        desc = (inc.get('Description') or '').lower()
        combined = raw_type + ' ' + desc
        matched = False
        for keyword, weight in severity_map.items():
            if keyword in combined:
                multiplier *= weight
                types_seen.append(keyword.replace(' ', '_'))
                matched = True
                break
        if not matched:
            multiplier *= 1.3
            types_seen.append(raw_type or 'unknown')
    label = types_seen[0] if types_seen else 'alert'
    return min(multiplier, 3.5), label

def _bus_deviation_to_delay_rate(mean_abs_deviation_min, baseline=DC_TRANSIT_BASELINE):
    """Normalise mean absolute bus deviation (minutes) to a [0, 0.95] delay-rate."""
    extra = (mean_abs_deviation_min / 20.0) * 0.40
    return min(baseline + extra, 0.95)

def build_transit(city_cfg, weather_df, days=30):
    """
    Today's row uses LIVE WMATA API data.
    Historical rows use WMATA DC baselines with weather/DoW/incident simulation.
    """
    rows = []
    dates = pd.date_range(end=datetime.now(), periods=days, freq='D')
    dow_mult = {0: 1.25, 1: 1.05, 2: 1.00, 3: 1.05, 4: 1.30, 5: 0.55, 6: 0.40}

    print('  [WMATA] Fetching live rail incidents...')
    live_incidents = _fetch_rail_incidents()
    live_inc_mult, live_inc_type = _incidents_to_stress(live_incidents)
    print('  [WMATA] Found', len(live_incidents), 'active rail incident(s)')

    print('  [WMATA] Fetching live bus deviations...')
    live_deviation = _fetch_bus_deviations()
    print('  [WMATA] Mean bus deviation:', round(live_deviation, 1), 'min')

    today = date.today()

    for dt in dates:
        is_today = (dt.date() == today)
        base = city_cfg.get('transit_baseline_delay', DC_TRANSIT_BASELINE)
        dow = dt.dayofweek

        w_row = weather_df[weather_df['date'].dt.date == dt.date()]
        if not w_row.empty:
            w_factor = 1.0 + w_row.iloc[0]['rain_stress'] * 1.2 + w_row.iloc[0]['wind_stress'] * 0.4
        else:
            w_factor = 1.0

        if is_today:
            incident = live_inc_mult
            incident_type = live_inc_type if live_incidents else 'none'
            if live_deviation > 0:
                delay_rate = _bus_deviation_to_delay_rate(live_deviation, base)
                delay_rate = min(delay_rate * dow_mult.get(dow, 1.0) * w_factor, 0.95)
            else:
                delay_rate = min(base * dow_mult.get(dow, 1.0) * w_factor * incident, 0.95)
        else:
            if random.random() < 0.09:
                incident = random.uniform(1.4, 2.2)
                incident_type = random.choice(['signal_failure', 'track_work', 'overcrowding', 'medical_emergency'])
            else:
                incident = 1.0
                incident_type = 'none'
            delay_rate = min(base * dow_mult.get(dow, 1.0) * w_factor * incident, 0.95)

        rush_factor = 1.6 if dow < 5 else 0.7
        crowd_index = min((city_cfg['population_density'] / 10000) * rush_factor * random.uniform(0.85, 1.15), 1.0)

        rows.append({
            'date': dt, 'day_of_week': dt.strftime('%A'),
            'delay_rate': round(delay_rate, 4), 'crowd_index': round(crowd_index, 4),
            'incident_type': incident_type,
            'transit_stress': round(delay_rate * 0.65 + crowd_index * 0.35, 4),
            'data_source': 'wmata_live' if is_today else 'wmata_calibrated_sim',
        })

    return pd.DataFrame(rows)


cfg['population_density'] = cfg.get('population_density', 4400)
cfg['transit_baseline_delay'] = DC_TRANSIT_BASELINE

df_transit = build_transit(cfg, df_weather)
print('Transit:', len(df_transit), 'days | Avg delay:', round(df_transit['delay_rate'].mean() * 100, 1), '%')
print('Incidents:', int((df_transit['incident_type'] != 'none').sum()), 'days')


In [ ]:
df_transit['date'] = pd.to_datetime(df_transit['date']).dt.normalize()
df_weather['date'] = pd.to_datetime(df_weather['date']).dt.normalize()

df_merged = df_transit.merge(df_weather[['date', 'rain_stress', 'wind_stress']], on='date', how='left')

corr_rain = df_merged['rain_stress'].corr(df_merged['delay_rate'])
corr_wind = df_merged['wind_stress'].corr(df_merged['delay_rate'])
print('Correlation with delay_rate:')
print('  Rain stress ->', round(corr_rain, 3))
print('  Wind stress ->', round(corr_wind, 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Weather vs Delay Rate', fontsize=14, fontweight='bold')
for ax, col, color, label in zip(
    axes, ['rain_stress', 'wind_stress'], ['steelblue', 'purple'],
    ['Rain Stress (r=%.2f)' % corr_rain, 'Wind Stress (r=%.2f)' % corr_wind]
):
    ax.scatter(df_merged[col], df_merged['delay_rate'], alpha=0.6, color=color)
    ax.set_xlabel(col)
    ax.set_ylabel('Delay Rate')
    ax.set_title(label)
    m = pd.Series(df_merged[col]).cov(df_merged['delay_rate']) / df_merged[col].var()
    b = df_merged['delay_rate'].mean() - m * df_merged[col].mean()
    x = pd.Series(sorted(df_merged[col].dropna()))
    ax.plot(x, m * x + b, color='red', linewidth=2, linestyle='--', label='trend')
    ax.legend()
plt.tight_layout()
plt.show()


## Step 5 -- Load All Signals into SQLite

In [ ]:
conn = sqlite3.connect('city_stress.db')
df_weather.to_sql('weather', conn, if_exists='replace', index=False)

if df_news.empty:
    print("df_news is empty -- creating 'news' table with the expected schema anyway.")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS news (
            date TEXT, source TEXT, title TEXT, compound REAL,
            pos REAL, neu REAL, neg REAL, matched_in TEXT, url TEXT
        )
    """)
else:
    df_news.to_sql('news', conn, if_exists='replace', index=False)

df_transit.to_sql('transit', conn, if_exists='replace', index=False)

print('Tables saved:')
for t in ['weather', 'news', 'transit']:
    n = pd.read_sql('SELECT COUNT(*) AS n FROM ' + t, conn).iloc[0, 0]
    print(' ', t, ':', n, 'rows')


In [ ]:
FUSION_SQL = '''
SELECT
    w.date,
    w.weather_label,
    w.temperature_2m_max        AS temp_max,
    w.precipitation_sum         AS rain_mm,
    w.windspeed_10m_max         AS wind_kph,
    w.weather_stress,
    w.heat_stress,
    w.rain_stress,
    w.wind_stress,
    COALESCE(n.headline_count,  0)    AS headline_count,
    COALESCE(n.avg_negativity,  0.2)  AS avg_negativity,
    COALESCE(n.avg_compound,    0.0)  AS avg_compound,
    COALESCE(n.sentiment_stress,0.3)  AS sentiment_stress,
    t.day_of_week,
    t.delay_rate,
    t.crowd_index,
    t.incident_type,
    t.transit_stress
FROM weather w
LEFT JOIN (
    SELECT
        DATE(date)                        AS date,
        COUNT(title)                      AS headline_count,
        AVG(neg)                          AS avg_negativity,
        AVG(compound)                     AS avg_compound,
        AVG((neg - pos + 1) / 2.0)        AS sentiment_stress
    FROM news
    GROUP BY DATE(date)
) n ON DATE(w.date) = n.date
LEFT JOIN transit t ON DATE(w.date) = DATE(t.date)
ORDER BY w.date
'''

df = pd.read_sql(FUSION_SQL, conn)
df['date'] = pd.to_datetime(df['date'])
print('Fused:', df.shape[0], 'rows x', df.shape[1], 'cols')
print('  Nulls:', df.isnull().sum()[df.isnull().sum() > 0].to_dict())
if (df['headline_count'] == 0).all():
    print('  NOTE: headline_count is 0 for every row -- news sentiment is using the')
    print('        fallback constant everywhere. Check NEWS_API_KEY and the keyword filter.')
df.tail(4)


## Step 6 -- Compute the City Stress Index (CSI 0-100)| Pillar | Weight | Rationale | Weather | 30% | Proven link between weather and urban anxiety | News Sentiment | 35% | Media negativity reflects and amplifies public mood | Transit | 35% | Daily commuter friction is the most direct stressor |

In [ ]:
W_WEATHER, W_SENTIMENT, W_TRANSIT = 0.30, 0.35, 0.35

df['raw_weather'] = (df['weather_stress']*0.50 + df['heat_stress']*0.25 +
                      df['rain_stress']*0.15   + df['wind_stress']*0.10)
df['raw_sentiment'] = df['sentiment_stress']
df['raw_transit'] = df['transit_stress']*0.70 + df['delay_rate']*0.30

df['csi_raw'] = (df['raw_weather']*W_WEATHER + df['raw_sentiment']*W_SENTIMENT + df['raw_transit']*W_TRANSIT)

scaler = MinMaxScaler(feature_range=(8, 92))
df['csi'] = scaler.fit_transform(df[['csi_raw']]).round(1)
df['csi_smoothed'] = df['csi'].rolling(3, center=True, min_periods=1).mean()

def risk_tier(s):
    if s < 30: return 'Calm'
    if s < 50: return 'Elevated'
    if s < 70: return 'High'
    return 'Critical'

def risk_color(s):
    if s < 30: return '#27AE60'
    if s < 50: return '#F39C12'
    if s < 70: return '#E67E22'
    return '#C0392B'

df['risk_tier'] = df['csi'].apply(risk_tier)
df['risk_color'] = df['csi'].apply(risk_color)
df.to_sql('city_stress_index', conn, if_exists='replace', index=False)

print('CSI computed and saved to SQLite')
print('Mean CSI :', round(df['csi'].mean(), 1))
print('Peak CSI :', round(df['csi'].max(), 1), 'on', df.loc[df['csi'].idxmax(), 'date'].strftime('%Y-%m-%d'))
print('Calm days:', int((df['risk_tier']=='Calm').sum()), '| High+Critical:', int(df['risk_tier'].isin(['High','Critical']).sum()))
df[['date','weather_label','risk_tier','csi','csi_smoothed']].tail(15)


## Step 7 -- SQL-Powered Business Insights

In [ ]:
queries = {
    'Stress by Day of Week': '''
        SELECT day_of_week,
               ROUND(AVG(csi),1) AS avg_stress,
               ROUND(MAX(csi),1) AS peak_stress,
               COUNT(*)          AS days
        FROM city_stress_index
        GROUP BY day_of_week
        ORDER BY avg_stress DESC''',

    'Stress by Weather Condition': '''
        SELECT weather_label,
               ROUND(AVG(csi),1) AS avg_stress,
               COUNT(*)          AS occurrences
        FROM city_stress_index
        GROUP BY weather_label
        ORDER BY avg_stress DESC LIMIT 8''',

    'Worst 5 Days': '''
        SELECT date, weather_label, incident_type,
               ROUND(csi,1) AS stress_index, risk_tier
        FROM city_stress_index
        ORDER BY csi DESC LIMIT 5''',

    'Stress During Transit Incidents': '''
        SELECT incident_type,
               ROUND(AVG(csi),1) AS avg_csi,
               COUNT(*) AS count
        FROM city_stress_index
        WHERE incident_type != "none"
        GROUP BY incident_type ORDER BY avg_csi DESC''',
}

for title, sql in queries.items():
    print()
    print('=' * 50)
    print(' ', title)
    print('=' * 50)
    print(pd.read_sql(sql, conn).to_string(index=False))


## Step 8 -- ML Forecasting: Predict Tomorrow's Stress Score Gradient Boosting Regressor with TimeSeriesSplit cross-validation. I never shuffle time data -- past must always train future.

In [ ]:
df_ml = df.copy().sort_values('date').reset_index(drop=True)

df_ml['dow'] = df_ml['date'].dt.dayofweek

for lag in [1, 2, 3]:
    df_ml[f'csi_lag_{lag}'] = df_ml['csi'].shift(lag)
    df_ml[f'transit_lag_{lag}'] = df_ml['transit_stress'].shift(lag)
    df_ml[f'sentiment_lag_{lag}'] = df_ml['sentiment_stress'].shift(lag)

df_ml['csi_roll3_mean'] = df_ml['csi'].shift(1).rolling(3).mean()

df_ml['is_monday'] = (df_ml['dow']==0).astype(int)
df_ml['is_friday'] = (df_ml['dow']==4).astype(int)

df_ml['target'] = df_ml['csi'].shift(-1)
df_ml.dropna(inplace=True)

FEATURES = [
    'csi_lag_1',
    'csi_roll3_mean',
    'raw_transit',
    'raw_sentiment',
    'is_monday',
    'is_friday',
]

X, y = df_ml[FEATURES], df_ml['target']

model = GradientBoostingRegressor(
    n_estimators=30,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=5,
    subsample=0.7,
    random_state=42,
)

tscv = TimeSeriesSplit(n_splits=5)
maes = []
for tr, va in tscv.split(X):
    model.fit(X.iloc[tr], y.iloc[tr])
    maes.append(mean_absolute_error(y.iloc[va], model.predict(X.iloc[va])))

model.fit(X, y)
print('TimeSeriesSplit CV MAE :', round(np.mean(maes), 2), 'CSI points (+/-', round(np.std(maes), 2), ')')
print('Final model MAE        :', round(mean_absolute_error(y, model.predict(X)), 2), 'CSI points')


In [ ]:
latest = df_ml.iloc[[-1]][FEATURES]
tomorrow_csi = round(float(model.predict(latest)[0]), 1)
tomorrow_tier = risk_tier(tomorrow_csi)
tomorrow_date = (datetime.now() + timedelta(days=1)).strftime('%A, %b %d')

advice = {
    'Calm':     'Normal conditions. Great day for outdoor events.',
    'Elevated': 'Minor disruptions possible. Add buffer to commute times.',
    'High':     'Significant friction forecast. Expect delays and crowds.',
    'Critical': 'Extreme conditions likely. Consider remote work or off-peak travel.',
}
print('=' * 46)
print('  FORECAST  |', TARGET_CITY.upper())
print(' ', tomorrow_date)
print('=' * 46)
print('  Predicted Stress Index :', tomorrow_csi, '/ 100')
print('  Risk Tier              :', tomorrow_tier)
print('  Advisory               :', advice[tomorrow_tier])
print('=' * 46)


In [ ]:
imp = pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_})
imp = imp.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp['feature'], imp['importance'], color='#2980B9', edgecolor='white')
ax.set_title('Gradient Boosting - Feature Importances', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', bbox_inches='tight')
plt.show()


## Step 9 -- Interactive 6-Panel Plotly Dashboard

In [ ]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        TARGET_CITY + ' -- 30-Day City Stress Index (CSI)',
        'Stress Pillar Breakdown (Stacked)',
        'Stress Distribution by Day of Week',
        'Temperature vs Stress (Rain = bubble size)',
        'Transit Delay Rate Timeline',
        'News Sentiment vs CSI',
    ),
    vertical_spacing=0.13, horizontal_spacing=0.10,
)

for lo, hi, col in [(0,30,'rgba(39,174,96,0.09)'),(30,50,'rgba(243,156,18,0.09)'),
                    (50,70,'rgba(230,126,34,0.09)'),(70,100,'rgba(192,57,43,0.09)')]:
    fig.add_hrect(y0=lo, y1=hi, fillcolor=col, line_width=0, row=1, col=1)

fig.add_trace(go.Scatter(x=df['date'], y=df['csi_smoothed'], mode='lines',
    name='CSI smoothed', line=dict(color='#2C3E50', width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['date'], y=df['csi'], mode='markers',
    name='Daily CSI', showlegend=False,
    marker=dict(color=df['risk_color'].tolist(), size=7, line=dict(width=1,color='white'))), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[datetime.now()+timedelta(days=1)], y=[tomorrow_csi],
    mode='markers+text', name='Tomorrow: ' + str(tomorrow_csi),
    marker=dict(symbol='star', size=18, color='#8E44AD'),
    text=['  ' + str(tomorrow_csi)], textposition='middle right'), row=1, col=1)

for col_name, w, color, fill in [
    ('raw_weather',   W_WEATHER,   '#3498DB', 'rgba(52,152,219,0.4)'),
    ('raw_sentiment', W_SENTIMENT, '#E74C3C', 'rgba(231,76,60,0.4)'),
    ('raw_transit',   W_TRANSIT,   '#F39C12', 'rgba(243,156,18,0.4)'),
]:
    name = col_name.replace('raw_','').title()
    fig.add_trace(go.Scatter(
        x=df['date'], y=(df[col_name]*w*100).round(1),
        stackgroup='p', name=name, line=dict(color=color), fillcolor=fill), row=1, col=2)

for dow in ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']:
    sub = df[df['day_of_week']==dow]
    fig.add_trace(go.Box(y=sub['csi'], name=dow[:3], showlegend=False,
        marker_color='#2980B9', boxmean=True), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df['temp_max'], y=df['csi'], mode='markers',
    marker=dict(size=(df['rain_mm']/df['rain_mm'].max()*25+6).round(0),
                color=df['csi'], colorscale='RdYlGn_r', showscale=False),
    text=df['weather_label'], name='Temp/Rain/CSI', showlegend=False), row=2, col=2)

fig.add_trace(go.Scatter(
    x=df['date'], y=(df['delay_rate']*100).round(1),
    mode='lines', fill='tozeroy', name='Delay %',
    line=dict(color='#E67E22'), fillcolor='rgba(230,126,34,0.25)', showlegend=False), row=3, col=1)

fig.add_trace(go.Scatter(
    x=df['avg_compound'], y=df['csi'], mode='markers',
    marker=dict(color=df['sentiment_stress'], colorscale='RdYlGn_r', size=8, showscale=False),
    name='Sentiment/CSI', showlegend=False), row=3, col=2)

fig.update_layout(
    height=960, width=1100,
    title_text='<b>City Stress Index Dashboard -- ' + TARGET_CITY + '</b>',
    title_font=dict(size=18, color='#2C3E50'),
    paper_bgcolor='#FAFAFA', plot_bgcolor='white',
    legend=dict(orientation='h', y=-0.05),
    font=dict(family='Arial', size=11),
)
fig.update_yaxes(title_text='CSI (0-100)', row=1, col=1)
fig.update_yaxes(title_text='Contribution', row=1, col=2)
fig.update_xaxes(title_text='Temp Max C', row=2, col=2)
fig.update_yaxes(title_text='CSI', row=2, col=2)
fig.update_yaxes(title_text='Delay %', row=3, col=1)
fig.update_xaxes(title_text='Compound Score', row=3, col=2)
fig.update_yaxes(title_text='CSI', row=3, col=2)

fig.write_html('outputs/city_stress_dashboard.html')
fig.show()
print('Dashboard saved to outputs/city_stress_dashboard.html')


In [ ]:
conn.close()
print('Project complete!')
import glob
print()
print('Output files:')
for f in sorted(glob.glob('outputs/**', recursive=True)):
    print(' ', f)
print('  city_stress.db')
